In [4]:
# Cell 01 | Load FAQ documents for persistent ingestion
# Purpose: Load the FAQ knowledge base that will be written into a persistent SQLite-backed search index.
# Key points: 本格重用 ingest.py 的 load_faq_data()；Data loading 與 search backend 分離，之後可替換 index 而不重寫資料來源邏輯。
# Execution: 需先完成 Lesson 8 的 ingest.py；執行後應載入完整 FAQ documents，數量目前預期約為 1401。

from ingest import load_faq_data


documents = load_faq_data()

print("Documents loaded:", len(documents))

Documents loaded: 1401


In [5]:
# Cell 02 | Filter the LLM Zoomcamp documents
# Purpose: Select only the LLM Zoomcamp FAQ entries that belong to the knowledge base used by this RAG assistant.
# Key points: Persistent ingestion 應明確控制寫入範圍；這裡利用 course metadata 過濾 documents，而不是把所有課程資料都寫入目前的 index。
# Execution: 需先完成 Cell 01；執行後 docs_llm 應只包含 course == "llm-zoomcamp" 的 documents。

docs_llm = [
    doc
    for doc in documents
    if doc["course"] == "llm-zoomcamp"
]

print("LLM Zoomcamp documents:", len(docs_llm))
print("\nFirst document:")
print(docs_llm[0])

LLM Zoomcamp documents: 139

First document:
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [2]:
# Cell 03 | Create the persistent SQLite search index
# Purpose: Create a SQLite-backed full-text search index that can persist FAQ documents across application restarts.
# Key points: MinSearch 將 index 保存在 RAM；TextSearchIndex 則將資料寫入 SQLite file，讓 ingestion 與 query process 可以分離。
# Execution: 需先安裝 sqlitesearch；第一次執行應建立 faq.db，並顯示目前 index 中的 document count。

from sqlitesearch import TextSearchIndex


sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

print("Documents currently indexed:", sqlite_index.count())

Documents currently indexed: 0


In [6]:
# Cell 04 | Ingest FAQ documents into the persistent index
# Purpose: Write the filtered LLM Zoomcamp FAQ documents into the SQLite-backed search index.
# Key points: Ingestion 將資料從來源載入 persistent knowledge base；資料寫入 faq.db 後，即使 Kernel 關閉也能保留。
# Execution: 需先完成 Cell 01、02、03；為避免重複資料，本格只允許在空的 index 上執行。

current_count = sqlite_index.count()

if current_count != 0:
    raise RuntimeError(
        f"Index already contains {current_count} documents. "
        "Stop ingestion to avoid duplicate records."
    )

total_documents = len(docs_llm)

for position, doc in enumerate(docs_llm, start=1):
    sqlite_index.add(doc)

    if position % 50 == 0 or position == total_documents:
        print(f"Ingested {position}/{total_documents} documents")

print("\nIngestion completed.")
print("Documents indexed:", sqlite_index.count())

Ingested 50/139 documents
Ingested 100/139 documents
Ingested 139/139 documents

Ingestion completed.
Documents indexed: 139


In [7]:
# Cell 05 | Close and verify the persistent database
# Purpose: Close the SQLite search connection and confirm that the persistent database file exists on disk.
# Key points: Persistent index 的資料儲存在 faq.db，而不是只存在 Python 記憶體；關閉 connection 不會刪除已寫入的 documents。
# Execution: 需先完成 Cell 04；執行後應確認 faq.db 存在，並顯示實際檔案大小。

from pathlib import Path


sqlite_index.close()

db_path = Path("faq.db")

print("Database exists:", db_path.exists())
print("Database path:", db_path.resolve())
print("Database size:", db_path.stat().st_size, "bytes")

Database exists: True
Database path: G:\01.Project\rag-knowledge-assistant\notebooks\faq.db
Database size: 401408 bytes
